### 2. DATA QUALITY ANALYSIS

**Goal:**  
Identify data quality issues and define a robust cleaning strategy for cohort and retention analysis.

**Why it matters:**  
Cohort analysis is highly sensitive to data inconsistencies (missing IDs, duplicates, returns).  
Incorrect preprocessing → misleading retention metrics.

In [2]:
# Set working directory
import local_config
from local_config import directory_path
import os
os.chdir(directory_path)

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Display settings
pd.set_option('display.max_columns', None)

# Load dataset
df = pd.read_excel('data/raw/OnlineRetail.xlsx')

---
### 1. Missing CustomerID

### Analysis

CustomerID is critical for user-level analytics (cohorts, retention, LTV).  
We investigate the scale and nature of missing values.

In [3]:
# Number of missing CustomerID
missing_customers = df['CustomerID'].isna().sum()
print(f"Missing CustomerID: {missing_customers}")

Missing CustomerID: 135080


In [4]:
# Unique invoices without CustomerID
df[df['CustomerID'].isna()]['InvoiceNo'].nunique()

3710

In [5]:
# Items per invoice (check if invoices are meaningful transactions)
df[df['CustomerID'].isna()] \
    .groupby('InvoiceNo') \
    .size() \
    .sort_values(ascending=False) \
    .head()

InvoiceNo
573585    1114
581219     749
581492     731
580729     721
558475     705
dtype: int64

In [6]:
# Distribution by country
df[df['CustomerID'].isna()]['Country'].value_counts().head()

Country
United Kingdom    133600
EIRE                 711
Hong Kong            288
Unspecified          202
Switzerland          125
Name: count, dtype: int64

In [7]:
# Share of missing CustomerID in UK
uk_df = df[df['Country'] == 'United Kingdom']

magnitude = uk_df['CustomerID'].isna().mean()
print(f"Share of missing CustomerID in UK: {magnitude:.2%}")

Share of missing CustomerID in UK: 26.96%


In [8]:
# Revenue share from missing CustomerID
missing_revenue = (
    df[df['CustomerID'].isna()]['Quantity'] *
    df[df['CustomerID'].isna()]['UnitPrice']
).sum()

total_revenue = (df['Quantity'] * df['UnitPrice']).sum()

print(f"Revenue share (missing CustomerID): {missing_revenue / total_revenue:.2%}")

Revenue share (missing CustomerID): 14.85%


### Business Insight

- Missing CustomerIDs represent **real transactions**, not technical errors  
- They are heavily concentrated in the **United Kingdom**
- These transactions contribute a **non-trivial share of revenue**
- Likely explanation: **guest checkout / unregistered users**

### Analytical Impact

Without CustomerID, we cannot:
- build cohorts
- calculate retention
- estimate LTV
- track repeat behavior

### Decision

- ❌ Exclude rows with missing `CustomerID` from cohort analysis  
- ✅ Optionally keep them for:
  - total revenue reporting
  - AOV (Average Order Value)

This ensures analytical correctness while preserving business visibility.

---
### 2. Returns & Negative Values

### Analysis

Negative quantities typically indicate product returns or corrections.
We validate this assumption.

In [9]:
df[df['Quantity'] < 0].head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom


In [10]:
# Share of negative quantities in invoices starting with "C"
negative_df = df[df['Quantity'] < 0]

share_c_invoices = (
    negative_df['InvoiceNo'].str.startswith('C', na=False).mean()
)

print(f"Share of negative qty with 'C' invoices: {share_c_invoices:.2%}")

Share of negative qty with 'C' invoices: 87.42%


### Key Observations

- Most negative quantities are linked to invoices starting with **"C"**
- However:
  - Not all "C" invoices contain only negative values
  - Not all negative values perfectly match original purchases

- Special case:
  - `StockCode = 'D'` → **Discounts**
  - Always negative, not actual returns

In [11]:
# Check if 'D' stock code has positive values
df[(df['StockCode'] == 'D') & (df['Quantity'] > 0)].shape[0]

0

### Business Insight

Negative transactions represent:

1. **Returns / refunds**
2. **Discount adjustments (StockCode = 'D')**
3. Possible **data inconsistencies** (missing matching purchase rows)

### Decision

- ✅ Keep negative quantities in dataset  
- ✅ Create new column:

```python
df['TransactionType'] = np.where(df['Quantity'] < 0, 'Return', 'Purchase')

For analysis:

Gross metrics → exclude returns

Net metrics → include returns

This allows flexible and more realistic business analysis.


---
### 3. Outliers

### Analysis

Outliers in `Quantity` and `UnitPrice` were previously explored.

### Business Insight

- Extreme values may represent:
  - bulk purchases (B2B behavior)
  - data entry anomalies

- Their share is relatively small and does not significantly distort aggregates

### Decision

- ✅ Keep outliers in dataset  
- ❗ Monitor during metric calculation (especially AOV, revenue)

Removing them could lead to loss of important business signals.

---
### 4. Duplicates
### Analysis

We check for exact and partial duplicates.

In [12]:
# Exact duplicates
duplicates = df.duplicated().sum()
print(f"Exact duplicate rows: {duplicates}")

Exact duplicate rows: 5268


In [13]:
# Duplicates by key transaction fields
dupl_key = df.duplicated(
    subset=['InvoiceNo', 'StockCode', 'Quantity', 'UnitPrice'],
    keep=False
).sum()

print(f"Duplicates (key fields): {dupl_key}")

Duplicates (key fields): 10153


In [14]:
# Potential repeated items in same invoice
df.groupby(['InvoiceNo', 'StockCode']) \
  .size() \
  .reset_index(name='count') \
  .query('count > 1') \
  .head()

,InvoiceNo,StockCode,count
131,536381,71270,2
490,536409,21866,2
498,536409,22111,2
509,536409,22866,2
510,536409,22900,2


### Business Insight

- Some duplicates may represent:
  - real multiple quantities split into rows
  - system duplication errors

- Exact duplicates are most likely **data issues**

### Decision

- ❌ Remove exact duplicate rows  
- ⚠️ Keep multi-line invoice entries (valid business behavior)

```python
df = df.drop_duplicates()


---

### Final Cleaning Strategy

Based on the analysis, the following preprocessing steps will be applied:

### 1. Customer Filtering
- Remove rows with missing `CustomerID`

### 2. Transaction Handling
- Keep returns (negative Quantity)
- Add `TransactionType` column

### 3. Duplicates
- Remove exact duplicate rows

### 4. Outliers
- Keep as-is (monitor during analysis)

### 5. Data Type Optimization
- Convert:
  - `CustomerID` → integer / categorical
  - object columns → string

---

### Final Note

These decisions ensure:
- accurate cohort construction
- reliable retention metrics
- minimal loss of meaningful business data